# 05 - LLM Zero-Shot Baseline

Test LLM-based zero-shot classification using Groq (free tier) for Bangla toxic comment detection.

Strategy: batch 25 comments per request to stay within free tier limits (~1000 req/day on Groq).

In [ ]:
# Uncomment to install
# !pip install groq google-generativeai scikit-learn pandas pyyaml -q

In [ ]:
import os
import json
import time
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
import sys
sys.path.append('..')
from src.utils import load_config, save_metrics, ensure_dirs

config = load_config('../configs/config.yaml')
ensure_dirs('../results')

## Setup API Key

Set your Groq API key (free at https://console.groq.com):

In [ ]:
# Option 1: Set directly
# os.environ['GROQ_API_KEY'] = 'your-key-here'

# Option 2: Already in environment
assert os.environ.get('GROQ_API_KEY'), 'Set GROQ_API_KEY environment variable'

from groq import Groq
client = Groq()
LLM_MODEL = config['llm']['groq']['model']
BATCH_SIZE = config['llm']['groq']['batch_size']
print(f'Using model: {LLM_MODEL}, batch size: {BATCH_SIZE}')

## Load Test Data

In [ ]:
DATA_PATH = '../data/test.csv'
TEXT_COL = config['data']['text_column']
LABEL_COL = config['data']['label_column']

df = pd.read_csv(DATA_PATH)
texts = df[TEXT_COL].astype(str).tolist()
labels = df[LABEL_COL].astype(int).tolist()
print(f'Loaded {len(texts)} test samples')

## Batch Classification

In [ ]:
def classify_batch(batch_texts, client, model):
    numbered = '\n'.join(f'{i+1}. {t}' for i, t in enumerate(batch_texts))
    prompt = f"""You are a Bangla toxic comment classifier. For each numbered comment below,
respond with ONLY the number followed by 0 (non-toxic) or 1 (toxic).

Example output format:
1. 0
2. 1
3. 0

Comments:
{numbered}
"""
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
        max_tokens=len(batch_texts) * 10,
    )
    
    result_text = response.choices[0].message.content.strip()
    predictions = []
    for line in result_text.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        try:
            label = int(line.split('.')[-1].strip().split()[0])
            predictions.append(1 if label == 1 else 0)
        except (ValueError, IndexError):
            predictions.append(0)
    
    while len(predictions) < len(batch_texts):
        predictions.append(0)
    return predictions[:len(batch_texts)]

In [ ]:
all_predictions = []
delay = config['llm']['groq'].get('delay_between_batches', 2)

start_time = time.time()
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    total_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE
    
    try:
        preds = classify_batch(batch, client, LLM_MODEL)
        all_predictions.extend(preds)
        print(f'  Batch {batch_num}/{total_batches}: {len(preds)} predictions')
    except Exception as e:
        print(f'  Batch {batch_num} failed: {e}')
        all_predictions.extend([0] * len(batch))
    
    if i + BATCH_SIZE < len(texts):
        time.sleep(delay)

inference_time = time.time() - start_time
print(f'\nTotal time: {inference_time:.1f}s')
print(f'Predicted toxic: {sum(all_predictions)} / {len(all_predictions)}')

## Evaluation

In [ ]:
effective_labels = labels[:len(all_predictions)]

metrics = {
    'model': f'LLM ({LLM_MODEL})',
    'accuracy': accuracy_score(effective_labels, all_predictions),
    'f1': f1_score(effective_labels, all_predictions, average='weighted'),
    'precision': precision_score(effective_labels, all_predictions, average='weighted'),
    'recall': recall_score(effective_labels, all_predictions, average='weighted'),
    'inference_time': inference_time,
    'samples_per_second': len(all_predictions) / inference_time,
}

print('\n=== LLM Baseline Results ===')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

print('\n=== Classification Report ===')
print(classification_report(effective_labels, all_predictions, target_names=['non-toxic', 'toxic']))

print('\n=== Confusion Matrix ===')
print(confusion_matrix(effective_labels, all_predictions))

save_metrics(metrics, '../results/llm_baseline_results.json')
print('\nResults saved.')